# 第 72 课：推理队列与微批处理
## 最大排队时间、Prefill/Decode、EOS、取消与动态入队

第 71 课完成 Adapter 版本固定、灰度与回滚。本课再向服务内部走一步：**把不同到达时间、不同版本和不同回答长度的请求，放进同一个推理调度循环**。

本课独立实现两层内容：

1. **真实的随机微型 Transformer + Q/V LoRA**：使用 PyTorch 对右补齐的不同长度输入做 batched forward，验证与逐请求 greedy 输出一致。
2. **确定性离散 tick 调度器**：对比 static cohort 与 continuous batching，处理入队、逻辑 prefill、decode、EOS、长度、取消、deadline、排队超时、队列背压和版本切换。

**重要边界：这是可复现实验，不是网络推理服务器或 GPU 基准。** 没有真实 HTTP、CUDA 并发、PagedAttention、KV cache、CUDA Graph 或网络输出背压。Decode 为保证可见性，**每一步重新计算完整上下文**；逻辑 token budget 不是实际 PyTorch 计算量，所有 TTFT/E2E 单位是虚拟 tick，不是毫秒。模型未经过预训练，输出内容不代表真实回答能力。

## 0. 如何运行

只需已有 Python、PyTorch、Jupyter。**不联网、不下载模型**。从新内核按顺序执行；每次完整演练会创建 `lesson_72_outputs/run_时间戳/`，不会覆盖之前结果。

同名 `.py` 是完全独立版本，命令：

```bash
python lesson_72_inference_queue_microbatch_prefill_decode.py --checks-only
python lesson_72_inference_queue_microbatch_prefill_decode.py
python lesson_72_inference_queue_microbatch_prefill_decode.py --demo-only
python lesson_72_inference_queue_microbatch_prefill_decode.py --reload-run lesson_72_outputs/具体运行目录
```

默认 CPU 且只有一个 PyTorch CPU 线程，适合练习调度正确性；未实测 RTX 3060 显存、吞吐或真实 TTFT。完整演练与自动检查都在 Notebook 末尾开启，独立练习开关默认关闭。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第72课：推理队列与微批处理——逻辑调度、prefill/decode、EOS、取消与背压。

独立离线实现：用真实小型因果Transformer/QV-LoRA验证批量前向的正确性；
以确定性逻辑时钟模拟静态/连续批处理、请求级版本固定、超时及取消。
无HTTP、无真实服务并发、无KV cache，逻辑吞吐不是 GPU 吞吐或真实毫秒延迟。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import platform
import random
import re
import tempfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；本课不自动安装或下载模型。") from exc

VERSION = "lesson72-scheduler-v1"
SYSTEM_PROMPT = "Reply with one English word."
SOURCES = {
 "continuous_batching": "https://huggingface.co/docs/transformers/continuous_batching",
 "tgi_router": "https://github.com/huggingface/text-generation-inference/blob/main/router/README.md",
 "vllm_scheduler": "https://docs.vllm.ai/en/latest/api/vllm/config/scheduler/",
 "vllm_internals": "https://vllm-project.github.io/2025/09/05/anatomy-of-vllm.html",
 "inference_mode": "https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html",
 "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
seed_all(7201)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA可用:', torch.cuda.is_available())
print('计时方式：确定性逻辑tick，不是wall-clock毫秒')

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
计时方式：确定性逻辑tick，不是wall-clock毫秒


## 1. 先分清三件事

**Prefill**：输入完整提示，计算第一个待生成位置的分布；教学调度器记一次逻辑工作量为 `len(prompt_token_ids)`。

**Decode**：根据已经生成的前缀继续产生下一个 token；若有真正的 KV cache，一般只需处理新位置。本课**没有 KV cache**，所以实际 PyTorch 前向依然输入全部上下文，但调度预算为了展示调度概念，把该步计作 1 个逻辑新 token。

**Microbatch**：把同一步需要计算的多个请求打包成张量；不同 prompt 长度时右补齐，并在每行的**最后有效位置**读取 logits，而不是固定取最后一列。

工业级 continuous batching 会随着生成进度移出完成的请求、新请求可以在后续步加入。Hugging Face 的 [continuous batching 文档](https://huggingface.co/docs/transformers/continuous_batching) 与 [TGI Router 说明](https://github.com/huggingface/text-generation-inference/blob/main/router/README.md) 描述了这种核心机制。本课只还原可审计的调度规则，不宣称使用了它们的实际后端。

## 2. 教学 tokenizer 与模型

复用上一课概念，但复制全部必要源码，不导入前课。特殊 ID（BOS、EOS、`im_start` 等）单独编码；用户文字里的 `<im_start>` **不会自动变成结构 token**。真实模型迁移时必须使用其官方 tokenizer 与 chat template，这份 UTF-8 字节＋少量 merge 的实现不兼容任何指定的 HF 模型权重。

小模型是带因果注意力的真实 PyTorch Transformer。两个版本来自同一随机基座，各自有独立 Q/V LoRA；权重冻结，按版本分组前向，**不在请求运行途中修改共享模型权重**。

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')

class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)

class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))

class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)

def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy

def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]

def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}

def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))

def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base

def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [4]:
SAFE_ID = re.compile(r'^[a-z][a-z0-9_-]{0,39}$')
ALLOWED_OUTPUT = "abcdefghijklmnopqrstuvwxyz0123456789 .,!?"


def build_models(seed: int=7201):
    """同一随机基座 + 两份独立Q/V LoRA。版本只是不可变教学对象。"""
    require(type(seed) is int and seed >= 0, 'seed非法')
    seed_all(seed)
    tok = ByteMergeChatTokenizer()
    cfg = ToyConfig(tok.vocab_size, dim=32, heads=4, layers=2, ff_dim=64, max_length=192)
    base = TinyChatLM(cfg)
    a = inject_lora(base, rank=4, alpha=8.0)
    b = inject_lora(base, rank=4, alpha=8.0)
    # B=0时LoRA作用为零；通过不同adapter B制造第二个可核对的版本。
    generator = torch.Generator().manual_seed(seed+1)
    with torch.no_grad():
        for _,param in adapters(b):
            if param is not None and param.ndim == 2 and param.shape[1] == 4:  # B:[out,rank]
                param.copy_(torch.randn(param.shape, generator=generator)*0.12)
    models={'v0001':a.requires_grad_(False).eval(),
            'v0002':b.requires_grad_(False).eval()}
    require(all(not p.requires_grad for m in models.values() for p in m.parameters()),
            '服务模型必须全部冻结')
    require(base_hash(models['v0001']) == base_hash(models['v0002']),
            '两个版本必须来自同一基座')
    require(state_hash(models['v0001'].state_dict()) != state_hash(models['v0002'].state_dict()),
            '版本权重应该不同')
    return tok,models


class ModelRunner:
    """CPU FP32真实因果模型；每个调度步在同版本内部进行一次右补齐前向。

    为保持独立、方便逐token核对，decode时每次重算完整上下文，**没有KV cache**。
    逻辑调度的token budget是理想化的KV-cache工作量，不能当作实际PyTorch FLOPs。
    """
    def __init__(self,tok:ByteMergeChatTokenizer,models:dict):
        require(set(models)=={'v0001','v0002'},'需要固定两个模型版本')
        self.tok=tok;self.models=models
        self.cfg=next(iter(models.values())).cfg
        require(all(m.cfg==self.cfg for m in models.values()),'版本模型结构不一致')
        self.alphabet=[tok.eos_token_id]+[tok.byte_offset+ord(c) for c in ALLOWED_OUTPUT]
        require(len(set(self.alphabet))==len(self.alphabet),'输出动作重复')
        self.calls=0;self.forward_rows=0;self.forward_padded_positions=0
        self.forward_context_positions=0
        self.calls_by_version=Counter()

    def model_hashes(self):
        return {v:state_hash(m.state_dict()) for v,m in self.models.items()}

    def batch_next(self,version:str,contexts:list[list[int]]) -> list[int]:
        require(version in self.models,'请求的已固定版本不存在')
        require(type(contexts) is list and bool(contexts),'不能执行空batch')
        lengths=[len(c) for c in contexts]
        require(all(type(c) is list and 2<=len(c)<=self.cfg.max_length and
                    all(type(t) is int and 0<=t<self.tok.vocab_size for t in c)
                    for c in contexts),'输入ID/长度非法')
        width=max(lengths)
        ids=torch.full((len(contexts),width),self.tok.pad_token_id,dtype=torch.long)
        mask=torch.zeros_like(ids,dtype=torch.bool)
        for i,c in enumerate(contexts):
            ids[i,:len(c)]=torch.tensor(c)
            mask[i,:len(c)]=True
        with torch.inference_mode():
            logits=self.models[version](ids,mask)
            last=logits[torch.arange(len(contexts)),torch.tensor(lengths)-1]
            choice=last[:,self.alphabet].argmax(-1)
            tokens=[self.alphabet[int(j)] for j in choice]
        self.calls+=1;self.calls_by_version[version]+=1
        self.forward_rows+=len(contexts)
        self.forward_context_positions+=sum(lengths)
        self.forward_padded_positions+=len(contexts)*width
        return tokens

    def solo(self,version:str,prefix:list[int],max_new_tokens:int) -> list[int]:
        require(type(max_new_tokens) is int and 1<=max_new_tokens<=16,'生成预算非法')
        ids=prefix[:]; result=[]
        for _ in range(max_new_tokens):
            next_token=self.batch_next(version,[ids])[0]
            result.append(next_token)
            if next_token == self.tok.eos_token_id:break
            ids.append(next_token)
        return result


def text_from_tokens(tok:ByteMergeChatTokenizer,generated_ids:list[int]) -> str:
    require(type(generated_ids) is list and all(type(x) is int for x in generated_ids),
            '输出token必须是整数列表')
    visible=generated_ids[:-1] if generated_ids and generated_ids[-1]==tok.eos_token_id else generated_ids
    require(tok.eos_token_id not in visible,'EOS后不得继续生成')
    return tok.decode(visible,skip_special_tokens=False)

In [5]:
tok, models = build_models(7201)
runner = ModelRunner(tok, models)
print('词表大小:',tok.vocab_size)
print('模型参数量（每份）：',sum(p.numel() for p in models['v0001'].parameters()))
print('版本基座同一身份:',base_hash(models['v0001'])==base_hash(models['v0002']))
print('版本整体不同:',runner.model_hashes()['v0001']!=runner.model_hashes()['v0002'])
print('全部参数冻结:', all(not p.requires_grad for m in models.values() for p in m.parameters()))

词表大小: 281
模型参数量（每份）： 42048
版本基座同一身份: True
版本整体不同: True
全部参数冻结: True


## 3. 右补齐为什么必须按每行有效长度取 logits？

假设两条输入长度是 4 和 6：

```text
行0：[a b c d PAD PAD] → 要读取 d 所在位置的 logits
行1：[a b c d e f]     → 要读取 f 所在位置的 logits
```

正确做法是 `logits[batch_indices, lengths - 1]`，而不是 `logits[:, -1]`。Attention mask 防止把右侧 padding 当成 key，但**它不会自动让输出张量最后一列变成每条有效序列的最后一列**。

此外，我们采用确定性的 greedy，并限制可生成 token 为 EOS 与简单可打印 ASCII。这样在没有模型训练的情况下，真实模型输出也不会是未完结的 UTF-8 字节，从而避免把 tokenizer 错误混入批处理正确性。这个输出过滤本身就是教学采样约定，不能与未经处理的词表分布混称。

In [6]:
messages_a=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':'demo'}]
messages_b=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':'longer user demo'}]
a = tok.apply_chat_template(messages_a,tokenize=True,add_generation_prompt=True)
b = tok.apply_chat_template(messages_b,tokenize=True,add_generation_prompt=True)
runner = ModelRunner(tok,models)
batched = runner.batch_next('v0001',[a,b])
independent = [ModelRunner(tok,models).batch_next('v0001',[p])[0] for p in (a,b)]
print('前缀长度:',len(a),len(b))
print('batch输出ID:',batched)
print('逐条输出ID:',independent)
print('严格相同:',batched==independent)
print('batch内有效/包含pad的tensor位置:',runner.forward_context_positions,runner.forward_padded_positions)

前缀长度: 41 53
batch输出ID: [119, 62]
逐条输出ID: [119, 62]
严格相同: True
batch内有效/包含pad的tensor位置: 94 106


## 4. 每个请求必须有明确状态与预算

输入 `RequestSpec` 包含有界请求 ID、消息文本、到达 tick、最大新 token 数，以及可选取消 tick、绝对 deadline tick 和明确指定的版本。状态机为：

```text
到达 ──> queued ──> active ──> finished (eos 或 length)
             │          │
             └─> cancelled / timeout
             └─> rejected (queue_full: 接收之前)
```

具体实现把 `rejected` 作为直接终态，不会走 `queued→active`。一次接受固定版本与 revision；**无论在队列中等多久、稳定版本后来怎么切换，都不能中途改动这个请求的版本**。

最大队列 `max_queued` 是背压上限；`max_active` 限制同一步活跃请求数；`logical_token_budget` 限制每步计划工作量。请求过长且**没有实现分块 prefill**时，会提前拒绝，而不是偷偷截断。

In [7]:
@dataclass(frozen=True)
class RequestSpec:
    request_id: str
    user_text: str
    arrival_tick: int
    max_new_tokens: int = 5
    cancel_tick: int|None = None
    deadline_tick: int|None = None
    force_version: str|None = None

    def validate(self):
        require(type(self.request_id) is str and SAFE_ID.fullmatch(self.request_id),
                'request_id必须是短ASCII slug')
        bounded_str(self.user_text,96,'用户文本')
        require(bool(self.user_text),'用户文本不能为空')
        require(type(self.arrival_tick) is int and self.arrival_tick>=0,'到达tick非法')
        require(type(self.max_new_tokens) is int and 1<=self.max_new_tokens<=16,
                '最大输出token数非法')
        require(self.cancel_tick is None or
                (type(self.cancel_tick) is int and self.cancel_tick>=self.arrival_tick),
                '取消tick非法')
        require(self.deadline_tick is None or
                (type(self.deadline_tick) is int and self.deadline_tick>self.arrival_tick),
                '截止tick必须晚于到达tick')
        require(self.force_version in (None,'v0001','v0002'),'固定版本非法')


@dataclass(frozen=True)
class SchedulerConfig:
    max_active: int=3
    max_queued: int=12
    logical_token_budget: int=155
    max_batch_wait: int=1
    max_queue_wait: int=20
    target_prefill_batch: int=3
    max_ticks: int=120

    def validate(self):
        for name in ('max_active','max_queued','logical_token_budget',
                     'max_queue_wait','target_prefill_batch','max_ticks'):
            v=getattr(self,name)
            require(type(v) is int and v>0,name+'必须为正整数')
        require(type(self.max_batch_wait) is int and self.max_batch_wait>=0,
                'batch等待上限必须为非负整数')
        require(self.target_prefill_batch<=self.max_active,'目标聚合数不能大于active容量')
        require(self.logical_token_budget>=self.max_active,
                'budget应能同时供给已准入请求的每tick decode')


@dataclass
class RequestState:
    spec: RequestSpec
    prefix: list[int]
    pinned_version: str
    route_revision: int
    arrival_index: int
    status: str='queued'
    prefilled: bool=False
    first_prefill_tick: int|None=None
    first_token_tick: int|None=None
    terminal_tick: int|None=None
    stop_reason: str|None=None
    tokens: list[int]=field(default_factory=list)

    def public(self,tok):
        queue_wait=(self.first_prefill_tick-self.spec.arrival_tick
                    if self.first_prefill_tick is not None else None)
        ttft=(self.first_token_tick-self.spec.arrival_tick
              if self.first_token_tick is not None else None)
        return {'request_id':self.spec.request_id,'status':self.status,
                'version':self.pinned_version,'route_revision':self.route_revision,
                'arrival_tick':self.spec.arrival_tick,
                'first_prefill_tick':self.first_prefill_tick,
                'first_token_tick':self.first_token_tick,
                'terminal_tick':self.terminal_tick,'stop_reason':self.stop_reason,
                'queue_wait_ticks':queue_wait,'ttft_ticks':ttft,
                'e2e_ticks':(self.terminal_tick-self.spec.arrival_tick+1
                             if self.terminal_tick is not None else None),
                'generated_ids':self.tokens[:],
                'body':text_from_tokens(tok,self.tokens),
                'emitted_body_tokens':sum(t!=tok.eos_token_id for t in self.tokens)}


def fixture_specs() -> list[RequestSpec]:
    """全是合成输入；展示队列拥塞、批量、取消、排队超时、版本切换。"""
    workload=[]
    for i in range(20):
        tick=[0,0,0,1,1,2,2,2,3,3,4,4,5,5,6,7,8,9,10,11][i]
        workload.append(RequestSpec(
            request_id=f'req_{i:03d}',user_text=f'demo {i:02d}',arrival_tick=tick,
            max_new_tokens=3+(i%4),
            cancel_tick=(tick+2 if i in (5,10) else None),
            deadline_tick=(tick+5 if i in (6,16) else None),
            force_version=('v0001' if i==17 else None)))
    return workload


def nearest_rank(values:list[float],p:float) -> float|None:
    require(type(p) in (float,int) and 0<p<=1,'分位数p非法')
    if not values:return None
    sorted_values=sorted(values)
    return sorted_values[max(0,math.ceil(p*len(values))-1)]

In [8]:
example = RequestSpec('req_000','demo',arrival_tick=0,max_new_tokens=5,
                      cancel_tick=None,deadline_tick=8)
example.validate()
cfg=SchedulerConfig()
print('请求:',asdict(example))
print('调度配置:',asdict(cfg))
print('可用上下文上限:',runner.cfg.max_length)

请求: {'request_id': 'req_000', 'user_text': 'demo', 'arrival_tick': 0, 'max_new_tokens': 5, 'cancel_tick': None, 'deadline_tick': 8, 'force_version': None}
调度配置: {'max_active': 3, 'max_queued': 12, 'logical_token_budget': 155, 'max_batch_wait': 1, 'max_queue_wait': 20, 'target_prefill_batch': 3, 'max_ticks': 120}
可用上下文上限: 192


## 5. “最大凑批等待”与“最大实际排队时间”不是一个东西

本课的 `max_batch_wait` 表示：当活跃组为空、已有待处理请求但不足目标凑批数时，最多为了收集更多请求**主动等待**几个 tick。

这不是用户的服务排队时限！当所有活跃槽位已占满或 token 预算不够时，一个请求可能远远超过 `max_batch_wait` 仍排不上。真正控制排队超时的是 `max_queue_wait`；整体 deadline 则按绝对 tick 判断。

每个 tick，调度器先处理新版本路由、到达、取消和超时，**再**选择模型计算。这样取消发生的那个 tick 不再偷跑一个新 token；如果完成更早已经终止，迟到的取消则不会覆盖既有结果。

## 6. Static batching 与 continuous batching

**Static**：已开始的 cohort 没全部结束前，不允许其他 queued 请求加入，即使空出了部分槽位。

**Continuous**：有空槽和逻辑 token 预算时，新请求可在其他请求 decode 的同时 prefill；上一批 EOS 或 length 终止后马上释放槽位。不同模型版本仍要分组独立前向，不会错误地混用一组权重。

每步预算：

\[
\text{planned tokens} = \sum_{i\in\mathrm{new~prefill}} |x_i| + \#\mathrm{active~decode}
\]

这只是**教学逻辑预算**。真实运行因为没有 KV cache，decode 实际重新计算全部上下文，因此还记录 `dense_recomputed_tokens`，防止“逻辑 token 数少”被误认为实际计算量少。

In [9]:
class SimulatedScheduler:
    """逻辑tick模拟，不做网络线程。每个tick允许按版本进行若干真实batched forward。

    EOS与取消在tick边界处理；请求到达时固定模型版本和revision。
    `max_batch_wait`仅是主动凑批最长等待，不保证队列最多等待这么久。
    输出背压/网络断开、KV分配与真实GPU并发不在此模拟。
    """
    def __init__(self,runner:ModelRunner, config:SchedulerConfig, mode:str='continuous',
                 route_switches:dict[int,str]|None=None):
        config.validate()
        require(mode in ('continuous','static'),'调度模式非法')
        self.runner=runner;self.cfg=config;self.mode=mode
        self.route_switches=dict(route_switches or {})
        require(all(type(t) is int and t>=0 and v in runner.models
                    for t,v in self.route_switches.items()),'路由切换配置非法')
        self.current_route='v0001';self.route_revision=1
        self.states={};self.pending=[];self.active=[];self.events=[]
        self.work_batches=[]
        self.stats=Counter()
        self.last_tick=-1

    def event(self,tick:int,kind:str, request_id:str|None=None,**kwargs):
        require(kind in {'route_switch','accepted','rejected','prefill','decode','token',
                         'finished','cancelled','timeout'},'事件类型非法')
        # 白名单字段：永远不把原始user_text/提示放入调度追踪。
        allowed={'version','revision','reason','token_id','length','batch_id'}
        require(set(kwargs)<=allowed,'事件含未授权内容')
        self.events.append({'tick':tick,'kind':kind,'request_id':request_id,**kwargs})

    def _terminal(self,state:RequestState,tick:int,status:str,reason:str):
        require(state.status in {'queued','active'},'终态不可重复触发')
        require(status in {'finished','cancelled','timeout'},'终止类型非法')
        state.status=status;state.terminal_tick=tick;state.stop_reason=reason
        if state in self.pending:self.pending.remove(state)
        if state in self.active:self.active.remove(state)
        self.event(tick,status,state.spec.request_id,reason=reason,version=state.pinned_version)
        self.stats[status]+=1

    def _accept(self,spec:RequestSpec,index:int,tick:int):
        version=spec.force_version or self.current_route
        message=[{'role':'system','content':SYSTEM_PROMPT},
                 {'role':'user','content':spec.user_text}]
        prefix=self.runner.tok.apply_chat_template(message,tokenize=True,
                                                    add_generation_prompt=True)
        require(len(prefix)+spec.max_new_tokens<=self.runner.cfg.max_length,
                '请求超过模型上下文；必须在接受前拒绝，不能隐式截断')
        require(len(prefix)<=self.cfg.logical_token_budget,
                '完整prefill超出逻辑token预算；本课不实现chunked prefill')
        if len(self.pending)>=self.cfg.max_queued:
            result=RequestState(spec,prefix,version,self.route_revision,index,
                                status='rejected',terminal_tick=tick,stop_reason='queue_full')
            self.states[spec.request_id]=result
            self.event(tick,'rejected',spec.request_id,reason='queue_full',version=version)
            self.stats['rejected']+=1
            return
        result=RequestState(spec,prefix,version,self.route_revision,index)
        self.states[spec.request_id]=result;self.pending.append(result)
        self.stats['peak_pending']=max(self.stats['peak_pending'],len(self.pending))
        self.event(tick,'accepted',spec.request_id,version=version,revision=self.route_revision)
        self.stats['accepted']+=1

    def _expire_or_cancel(self,tick:int):
        for state in list(self.pending)+list(self.active):
            spec=state.spec
            if spec.cancel_tick is not None and tick>=spec.cancel_tick:
                self._terminal(state,tick,'cancelled','client_cancel')
            elif spec.deadline_tick is not None and tick>=spec.deadline_tick:
                self._terminal(state,tick,'timeout','deadline')
            elif state.status=='queued' and tick-spec.arrival_tick>=self.cfg.max_queue_wait:
                self._terminal(state,tick,'timeout','queue_timeout')

    def _admit(self,tick:int) -> list[RequestState]:
        if self.mode=='static' and self.active:
            return []
        if not self.pending:return []
        if not self.active and len(self.pending)<self.cfg.target_prefill_batch:
            oldest=self.pending[0].spec.arrival_tick
            if tick-oldest<self.cfg.max_batch_wait:
                return []
        budget=self.cfg.logical_token_budget-len(self.active)  # decode消耗逻辑1 token/请求
        slots=self.cfg.max_active-len(self.active)
        admitted=[]
        while self.pending and slots>0:
            state=self.pending[0]  # 严格FCFS；不跳过较长请求
            need=len(state.prefix)
            if need>budget:break
            self.pending.pop(0);self.active.append(state)
            state.status='active';state.first_prefill_tick=tick
            admitted.append(state)
            slots-=1;budget-=need
        return admitted

    def _forward_tick(self,tick:int,newly:list[RequestState]):
        if not self.active:return
        selected=list(self.active)
        require(len(selected)<=self.cfg.max_active,'active容量越界')
        new_ids={s.spec.request_id for s in newly}
        budget=sum((len(s.prefix) if s.spec.request_id in new_ids else 1) for s in selected)
        require(budget<=self.cfg.logical_token_budget,'本tick逻辑预算超出')
        self.stats['logical_scheduled_tokens']+=budget
        self.stats['prefill_requests']+=len(newly)
        self.stats['decode_requests']+=len(selected)-len(newly)
        self.stats['scheduled_request_steps']+=len(selected)
        self.stats['work_ticks']+=1
        self.stats['peak_active']=max(self.stats['peak_active'],len(selected))
        versions={version:[s for s in selected if s.pinned_version==version]
                  for version in self.runner.models}
        for version,group in versions.items():
            if not group:continue
            contexts=[s.prefix+s.tokens for s in group]
            emitted=self.runner.batch_next(version,contexts)
            require(len(emitted)==len(group),'runner输出batch大小不对')
            self.work_batches.append({'tick':tick,'version':version,'size':len(group),
                'prefill_count':sum(s.spec.request_id in new_ids for s in group),
                'decode_count':sum(s.spec.request_id not in new_ids for s in group),
                'logical_tokens':sum(len(s.prefix) if s.spec.request_id in new_ids else 1 for s in group),
                'dense_recomputed_tokens':sum(map(len,contexts))})
            for state,token in zip(group,emitted):
                require(type(token) is int and token in self.runner.alphabet,
                        'backend返回非法/非整数生成token')
                kind='prefill' if state.spec.request_id in new_ids else 'decode'
                state.prefilled=True
                if state.first_token_tick is None:state.first_token_tick=tick
                self.event(tick,kind,state.spec.request_id,version=version)
                state.tokens.append(token)
                if token!=self.runner.tok.eos_token_id:
                    self.event(tick,'token',state.spec.request_id,token_id=token)
                if token==self.runner.tok.eos_token_id:
                    self._terminal(state,tick,'finished','eos')
                elif len(state.tokens)>=state.spec.max_new_tokens:
                    self._terminal(state,tick,'finished','length')
        require(len(self.pending)<=self.cfg.max_queued,'pending容量越界')
        self.stats['peak_pending']=max(self.stats['peak_pending'],len(self.pending))

    def run(self,requests:list[RequestSpec]) -> dict:
        require(type(requests) is list and bool(requests),'需要非空请求列表')
        require(not self.states and not self.events,'同一调度器不可二次运行')
        for req in requests:req.validate()
        ids=[req.request_id for req in requests]
        require(len(set(ids))==len(ids),'request_id重复')
        # (arrival,index)排序保证同tick公平稳定；不采用字符串ID排序。
        schedule=sorted(enumerate(requests),key=lambda x:(x[1].arrival_tick,x[0]))
        pointer=0
        for tick in range(self.cfg.max_ticks):
            self.last_tick=tick
            if tick in self.route_switches:
                self.current_route=self.route_switches[tick]
                self.route_revision+=1
                self.event(tick,'route_switch',version=self.current_route,
                           revision=self.route_revision)
            while pointer<len(schedule) and schedule[pointer][1].arrival_tick==tick:
                index,req=schedule[pointer]
                self._accept(req,index,tick);pointer+=1
            self._expire_or_cancel(tick)  # 取消/超时先于本tick模型执行
            newly=self._admit(tick)
            self._forward_tick(tick,newly)
            self.stats['peak_pending']=max(self.stats['peak_pending'],len(self.pending))
            if pointer==len(schedule) and not self.pending and not self.active:
                break
        else:
            raise LessonError('超过max_ticks，检查工作负载是否永不终止；不伪造成功')
        require(len(self.states)==len(requests),'结果数与请求数不一致')
        require(not self.pending and not self.active,'退出时存在未结束请求')
        completed=[s for s in self.states.values() if s.status=='finished']
        ttft=[s.first_token_tick-s.spec.arrival_tick for s in completed]
        e2e=[s.terminal_tick-s.spec.arrival_tick+1 for s in completed]
        queued=[s.first_prefill_tick-s.spec.arrival_tick for s in self.states.values()
                if s.first_prefill_tick is not None]
        body_tokens=sum(t!=self.runner.tok.eos_token_id for s in self.states.values() for t in s.tokens)
        metrics={'requests':len(requests),'accepted':self.stats['accepted'],
                 'rejected':self.stats['rejected'],'finished':self.stats['finished'],
                 'cancelled':self.stats['cancelled'],'timeout':self.stats['timeout'],
                 'work_ticks':self.stats['work_ticks'],'total_ticks':self.last_tick+1,
                 'runner_forward_calls':self.runner.calls,'runner_forward_rows':self.runner.forward_rows,
                 'runner_dense_context_tokens':self.runner.forward_context_positions,
                 'runner_padded_positions':self.runner.forward_padded_positions,
                 'logical_scheduled_tokens':self.stats['logical_scheduled_tokens'],
                 'prefill_requests':self.stats['prefill_requests'],
                 'decode_requests':self.stats['decode_requests'],
                 'scheduled_request_steps':self.stats['scheduled_request_steps'],
                 'peak_active':self.stats['peak_active'],'peak_pending':self.stats['peak_pending'],
                 'mean_ttft_ticks':sum(ttft)/len(ttft) if ttft else None,
                 'p95_ttft_ticks':nearest_rank(ttft,.95),
                 'mean_e2e_ticks':sum(e2e)/len(e2e) if e2e else None,
                 'p95_e2e_ticks':nearest_rank(e2e,.95),
                 'p95_queue_wait_ticks':nearest_rank(queued,.95),
                 'body_tokens':body_tokens,
                 'body_tokens_per_logical_tick':body_tokens/(self.last_tick+1),
                 'meaning':'virtual ticks; no real milliseconds, GPU throughput, or KV-cache speedup'}
        return {'mode':self.mode,'config':asdict(self.cfg),
                'route_switches':{str(k):v for k,v in sorted(self.route_switches.items())},
                'model_hashes':self.runner.model_hashes(),
                'metrics':metrics,
                'results':[s.public(self.runner.tok) for s in
                           sorted(self.states.values(),key=lambda x:x.arrival_index)],
                'events':self.events,'work_batches':self.work_batches}

In [10]:
def verify_against_solo(output:dict, tok:ByteMergeChatTokenizer, models:dict,
                        requests:list[RequestSpec], factory=ModelRunner) -> dict:
    by_id={r.request_id:r for r in requests}
    runner=factory(tok,models)
    checked=0;partial=0
    for result in output['results']:
        if result['status']=='rejected':
            require(result['generated_ids']==[],'拒绝请求不能消耗生成token')
            continue
        spec=by_id[result['request_id']]
        message=[{'role':'system','content':SYSTEM_PROMPT},
                 {'role':'user','content':spec.user_text}]
        prefix=tok.apply_chat_template(message,tokenize=True,add_generation_prompt=True)
        expected=runner.solo(result['version'],prefix,spec.max_new_tokens)
        actual=result['generated_ids']
        require(actual==expected[:len(actual)],'批量/单独生成不一致、版本串线或padding取错位置')
        if result['status']=='finished':
            require(actual==expected,'已完成请求的生成token与逐请求参考不同')
            checked+=1
        else:
            partial+=1
    return {'completed_matches_solo':checked,'partial_prefix_matches_solo':partial,
            'rejected_produced_no_tokens':sum(r['status']=='rejected' for r in output['results']),
            'reference_forward_calls':runner.calls}


def event_invariants(result:dict,tok:ByteMergeChatTokenizer) -> dict:
    events=result['events']
    terminal={'finished','cancelled','timeout','rejected'}
    done=Counter()
    emitted=Counter()
    ended_tick={}
    for i,e in enumerate(events):
        require(type(e['tick']) is int and e['tick']>=0,'事件tick非法')
        if i:require(e['tick']>=events[i-1]['tick'],'事件时间倒序')
        rid=e['request_id']
        if e['kind'] in terminal:
            done[rid]+=1
            require(done[rid]==1,'终态重复')
            ended_tick[rid]=e['tick']
        elif e['kind']=='token':
            require(done[rid]==0,'终态之后流出了token')
            require(e['token_id']!=tok.eos_token_id,'EOS不能作为正文stream token')
            emitted[rid]+=1
    require(set(done)=={s['request_id'] for s in result['results']},'有请求缺终态')
    for r in result['results']:
        require(emitted[r['request_id']]==r['emitted_body_tokens'],'stream数量不等于已输出正文数量')
        require(ended_tick[r['request_id']]==r['terminal_tick'],'终态事件时间与结果不一致')
        if r['stop_reason']=='eos':
            require(r['generated_ids'] and r['generated_ids'][-1]==tok.eos_token_id,'EOS丢失')
            require(tok.eos_token_id not in r['generated_ids'][:-1],'EOS后生成了token')
        elif r['stop_reason']=='length':
            require(tok.eos_token_id not in r['generated_ids'],'错误地把EOS终止记成长度限制')
    return {'exactly_one_terminal_per_request':True,
            'no_stream_after_terminal':True,'eos_not_streamed_as_body':True,
            'stream_token_events':sum(emitted.values())}

## 7. 一个明确生成 EOS 的 stub，先测试服务协议

随机模型可能永远不发 EOS。为了不靠运气做生命周期测试，我们额外定义**只用于测试的确定性 ScriptedRunner**：每个请求先输出 `a`（版本2输出 `b`），第二步固定输出 EOS。

它不是训练模型的成绩，也不是一个 PyTorch 模型。真正的批量正确性仍由前面的实际 Transformer 检查。接下来用 stub 精确观察终态、取消顺序与版本切换。

In [11]:
class ScriptedRunner(ModelRunner):
    """只用于调度协议故障测试；**不作为真实模型实验结果**。"""
    def batch_next(self,version,contexts):
        require(version in self.models and bool(contexts),'runner参数错误')
        self.calls+=1;self.calls_by_version[version]+=1
        self.forward_rows+=len(contexts)
        self.forward_context_positions+=sum(map(len,contexts))
        self.forward_padded_positions+=len(contexts)*max(map(len,contexts))
        result=[]
        for context in contexts:
            markers=[i for i in range(len(context)-1) if
                     context[i]==self.tok.im_start_token_id and
                     context[i+1]==self.tok.role_ids['assistant']]
            require(len(markers)==1,'无法确定assistant前缀边界')
            prefix_len=markers[0]+2
            generated=len(context)-prefix_len
            if generated>=1:
                result.append(self.tok.eos_token_id)
            else:
                letter='a' if version=='v0001' else 'b'
                result.append(self.tok.byte_offset+ord(letter))
        return result


def fixture_demo(output_root=Path('lesson_72_outputs'),seed=7201):
    """默认离线展示：真实模型 + 逻辑队列，分别评估静态和连续调度。"""
    root=Path(output_root)
    root.mkdir(parents=True,exist_ok=True)
    folder=root/datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
    folder.mkdir(parents=False,exist_ok=False)
    tok,models=build_models(seed)
    cfg=SchedulerConfig()
    specs=fixture_specs()
    plan={5:'v0002',10:'v0001'}
    expected_hashes={v:state_hash(m.state_dict()) for v,m in models.items()}
    reports={};validations={}
    for mode in ('static','continuous'):
        runner=ModelRunner(tok,models)
        result=SimulatedScheduler(runner,cfg,mode,plan).run(specs)
        valid={'events':event_invariants(result,tok),
               'solo':verify_against_solo(result,tok,models,specs)}
        require(result['model_hashes']==expected_hashes,'实验期间模型参数被改动')
        require(result['metrics']['finished']+result['metrics']['cancelled']+
                result['metrics']['timeout']+result['metrics']['rejected']==len(specs),
                '最终结果计数不守恒')
        reports[mode]=result;validations[mode]=valid
        write_json(folder/(mode+'.json'),result)
    write_json(folder/'verification.json',validations)
    write_json(folder/'workload.json',{'requests':[asdict(s) for s in specs],
                                      'route_switches':plan,'scheduler_config':asdict(cfg)})
    # 权重用于信任域内只读重载；独立版本不执行热切换或训练。
    torch.save({'format':VERSION,'toy_config':asdict(models['v0001'].cfg),
                'tokenizer':tok.contract(),'seed':seed,
                'model_hashes':expected_hashes,
                'states':{v:{k:t.detach().cpu().clone() for k,t in model.state_dict().items()}
                          for v,model in models.items()}},folder/'models.pt')
    observed={'format':VERSION,'python':platform.python_version(),'torch':str(torch.__version__),
              'seed':seed,'device':'cpu','single_thread':torch.get_num_threads()==1,
              'models_are_random_initialized':True,
              'models':{'base_hash':base_hash(models['v0001']),
                        'versions':expected_hashes,
                        'trainable_parameters':sum(p.numel() for m in models.values()
                                                   for p in m.parameters() if p.requires_grad)},
              'metrics':{mode:report['metrics'] for mode,report in reports.items()},
              'boundaries':['logical_discrete_ticks_not_milliseconds',
                            'full_context_recompute_no_KV_cache',
                            'per_version_model_forward_not_actual_cross_version_fusion',
                            'no_HTTP_threads_cudagraph_or_GPU_benchmark',
                            'only_synthetic_inputs_no_personal_information',
                            'cache_budget_is_idealized_not_actual_compute']}
    write_json(folder/'summary.json',observed)
    # 新对象读取checkpoint核对权重与对应输出，不对原输出进行改写。
    reload=reload_run(folder)
    write_json(folder/'reload.json',reload)
    print('第72课离线队列演练:',folder,flush=True)
    for mode,result in reports.items():
        print(mode,'work_ticks',result['metrics']['work_ticks'],
              'finished',result['metrics']['finished'],
              'cancelled',result['metrics']['cancelled'],
              'timeout',result['metrics']['timeout'],
              'rejected',result['metrics']['rejected'],
              'TTFT p95 ticks',result['metrics']['p95_ttft_ticks'],flush=True)
    return folder


def reload_run(folder:Path)->dict:
    folder=Path(folder)
    blob=torch.load(folder/'models.pt',map_location='cpu',weights_only=True)
    require(blob['format']==VERSION,'checkpoint版本不匹配')
    tok=ByteMergeChatTokenizer()
    require(blob['tokenizer']==tok.contract(),'tokenizer契约不一致')
    seed=blob['seed'];new_tok,models=build_models(seed)
    require(new_tok.contract()==tok.contract(),'构建环境tokenizer变化')
    require(asdict(models['v0001'].cfg)==blob['toy_config'],'模型配置不兼容')
    for name,model in models.items():
        model.load_state_dict(blob['states'][name],strict=True)
        require(state_hash(model.state_dict())==blob['model_hashes'][name],
                '模型权重身份失配')
        model.eval().requires_grad_(False)
    workload=json.loads((folder/'workload.json').read_text(encoding='utf-8'))
    requests=[RequestSpec(**s) for s in workload['requests']]
    cfg=SchedulerConfig(**workload['scheduler_config'])
    plan={int(k):v for k,v in workload['route_switches'].items()}
    result={}
    for mode in ('static','continuous'):
        expected=json.loads((folder/(mode+'.json')).read_text(encoding='utf-8'))
        actual=SimulatedScheduler(ModelRunner(tok,models),cfg,mode,plan).run(requests)
        require(actual==expected,'独立重载后调度输出不同')
        result[mode]={'same_results_and_events':True,
                      'solo':verify_against_solo(actual,tok,models,requests),
                      'event_invariants':event_invariants(actual,tok)}
    return {'format':VERSION,'same_model_hashes':True,
            'trainable_parameters':sum(p.numel() for m in models.values()
                                       for p in m.parameters() if p.requires_grad),
            'modes':result,'note':'read-only replay; not a network service or latency benchmark'}

In [12]:
stub=ScriptedRunner(tok,models)
scripted_cfg=SchedulerConfig(max_active=2,max_queued=6,
                logical_token_budget=110,max_batch_wait=0,target_prefill_batch=2)
short = SimulatedScheduler(stub,scripted_cfg).run([
    RequestSpec('short','demo',0,3),
    RequestSpec('limited','demo',0,1),
])
for r in short['results']:
    print(r['request_id'],repr(r['body']),r['generated_ids'],r['stop_reason'],
          'first_token_tick=',r['first_token_tick'],'terminal_tick=',r['terminal_tick'])
print('事件顺序:',[(e['tick'],e['kind'],e['request_id']) for e in short['events']])

short 'a' [113, 2] eos first_token_tick= 0 terminal_tick= 1
limited 'a' [113] length first_token_tick= 0 terminal_tick= 0
事件顺序: [(0, 'accepted', 'short'), (0, 'accepted', 'limited'), (0, 'prefill', 'short'), (0, 'token', 'short'), (0, 'prefill', 'limited'), (0, 'token', 'limited'), (0, 'finished', 'limited'), (1, 'decode', 'short'), (1, 'finished', 'short')]


## 8. EOS、长度上限与取消是三种不同终止语义

- EOS：生成的 token 是特殊 EOS，**计入内部 `generated_ids`**；不作为正文 stream token。立即完成，不再 decode。
- length：已达到 `max_new_tokens`，而最后 token 不是 EOS。它是预算终止，不得假报 EOS。
- client_cancel：如果在 tick 开始时已请求取消，则**在这个 tick 的模型调用之前**终止。可以已有部分输出，但不得再发新 token。

本课没有网络流式输出，只产出事件序列。`token` 事件模拟普通正文 token 的推送。取消和 deadline 都在状态机中先处理，事件不含原始用户文本。

In [13]:
cancelled = SimulatedScheduler(ScriptedRunner(tok,models),scripted_cfg).run([
    RequestSpec('cancel','demo',0,3,cancel_tick=1),
])
print('取消结果:',cancelled['results'][0])
print('取消事件:',[(e['tick'],e['kind']) for e in cancelled['events']])
print('取消前是否只产生一个正文token:',cancelled['metrics']['body_tokens']==1)
print('事件审核:',event_invariants(cancelled,tok))

取消结果: {'request_id': 'cancel', 'status': 'cancelled', 'version': 'v0001', 'route_revision': 1, 'arrival_tick': 0, 'first_prefill_tick': 0, 'first_token_tick': 0, 'terminal_tick': 1, 'stop_reason': 'client_cancel', 'queue_wait_ticks': 0, 'ttft_ticks': 0, 'e2e_ticks': 2, 'generated_ids': [113], 'body': 'a', 'emitted_body_tokens': 1}
取消事件: [(0, 'accepted'), (0, 'prefill'), (0, 'token'), (1, 'cancelled')]
取消前是否只产生一个正文token: True
事件审核: {'exactly_one_terminal_per_request': True, 'no_stream_after_terminal': True, 'eos_not_streamed_as_body': True, 'stream_token_events': 1}


## 9. 什么时候会被拒绝？什么时候会排队超时？

队列容量满时，新的到达请求立即返回 `rejected/queue_full`，不参与模型计算，不能伪造成正常 EOS。后续若队列等待时间达到 `max_queue_wait`，状态转为 `timeout/queue_timeout`。

设置 `max_batch_wait=0` 也**不能承诺 TTFT=0**：活跃槽被别的请求占用时，后来者仍会等待。超时与拒绝必须单独纳入服务成功率/可用性统计，不可只计算已完成请求的响应速度。

本课约定：同 tick 多个新请求先依序进入等待队列，再选本步模型批次；因此队列容量可能使同 tick 的突发输入被拒绝，即使本步之后会释放槽位。这是本模拟器的明确接收语义。

In [14]:
queue_limit_cfg=SchedulerConfig(max_active=2,max_queued=1,
    logical_token_budget=110,max_batch_wait=0,target_prefill_batch=2)
pressure = SimulatedScheduler(ScriptedRunner(tok,models),queue_limit_cfg).run([
    RequestSpec(f'q{i}','demo',0,3) for i in range(4)
])
print('accepted/rejected:',pressure['metrics']['accepted'],pressure['metrics']['rejected'])
print('请求状态:',[(r['request_id'],r['status'],r['stop_reason']) for r in pressure['results']])

accepted/rejected: 1 3
请求状态: [('q0', 'finished', 'eos'), ('q1', 'rejected', 'queue_full'), ('q2', 'rejected', 'queue_full'), ('q3', 'rejected', 'queue_full')]


## 10. 版本切换：排队中的旧请求仍属旧版本

如果 tick 1 将默认稳定版本从 v0001 切到 v0002：

- tick 0 已接受的请求一直绑定 v0001；
- tick 1 才进入的请求绑定 v0002；
- 即使两个请求在同一步运行，runner 也按版本拆分为不同的模型前向；
- 强制版本的请求只改变当次请求归属，不修改全局稳定路由。

这延续上一课的请求级快照，而不是把权重在全局对象上反复热切换。每个版本是一份独立模型副本，资源成本高；生产中应使用支持多 adapter 并发的专门后端。

In [15]:
switched=SimulatedScheduler(ScriptedRunner(tok,models),scripted_cfg,
                      route_switches={1:'v0002'}).run([
    RequestSpec('old','demo',0,3), RequestSpec('new','demo',1,3)
])
print('固定版本与生成:',[(r['request_id'],r['version'],r['route_revision'],r['body'])
                        for r in switched['results']])
print('路由切换事件:',[e for e in switched['events'] if e['kind']=='route_switch'])

固定版本与生成: [('old', 'v0001', 1, 'a'), ('new', 'v0002', 2, 'b')]
路由切换事件: [{'tick': 1, 'kind': 'route_switch', 'request_id': None, 'version': 'v0002', 'revision': 2}]


## 11. 现在运行真正的微型 Transformer 模型与20条合成请求

默认工作负载包含不同时间到达、不同最大输出长度、两次取消、两次请求 deadline、两次稳定版本切换和一个强制旧版请求。静态与连续采用同样的权重、请求和规则。所有模型权重冻结。

**评估必须交叉核对每条回答的 token ID：** 对每条完成请求，在请求的固定版本上重新逐请求 greedy；对取消/超时请求比较其已输出前缀。如果同一固定版本因 batch 编组而改变生成内容，优先怀疑右 padding 的最后有效位置、EOS 处理或权重串线。

示例工作负载是合成的，**不含真实用户资料**。`workload.json` 为再现实验而保留这些合成文本；事件日志本身不保存原文。

In [16]:
workload = fixture_specs()
plan = {5:'v0002',10:'v0001'}
results = {}
for mode in ('static','continuous'):
    local_runner=ModelRunner(tok,models)
    report=SimulatedScheduler(local_runner,SchedulerConfig(),mode,plan).run(workload)
    results[mode]=report
    print(mode, {k:report['metrics'][k] for k in
          ('finished','cancelled','timeout','rejected','work_ticks','p95_ttft_ticks')})
    print('事件检查:',event_invariants(report,tok)['no_stream_after_terminal'])
    print('逐请求独立参考:',verify_against_solo(report,tok,models,workload)['completed_matches_solo'])

static {'finished': 16, 'cancelled': 2, 'timeout': 2, 'rejected': 0, 'work_ticks': 35, 'p95_ttft_ticks': 19}
事件检查: True


逐请求独立参考: 16


continuous {'finished': 16, 'cancelled': 2, 'timeout': 2, 'rejected': 0, 'work_ticks': 27, 'p95_ttft_ticks': 11}
事件检查: True


逐请求独立参考: 16


## 12. 读懂 TTFT、E2E、吞吐量与调度预算

| 指标 | 本课定义 | 不能冒称 |
|---|---|---|
| TTFT | 首次产生 token 的 tick − 到达 tick，**即使首 token 为 EOS 也有 TTFT** | 真实毫秒 TTFT |
| E2E | 终止 tick − 到达 tick + 1 | 网络请求实测延迟 |
| p95 | 对**已完成请求**的指标用 nearest-rank 计算 | 完整服务 SLO（取消/超时仍需单列） |
| body tokens/tick | 正文 token 总数 / 模拟总 tick | GPU 实测 tokens/s |
| logical token budget | 新请求预填长度＋已有请求每步1 token | 本代码的真实计算 token 量 |
| dense recompute | 实际 runner 每次完整上下文的有效 token 数合计 | KV cache 节省的计算量 |

**默认 CPU 虚拟时钟的本次结果：** 静态调度用了 35 个工作 tick，已完成请求 p95 TTFT 为 19 tick；连续调度分别是 27 tick、11 tick。两者均有 16 完成、2 取消、2 timeout。**这不是工业系统快了 35/27 倍或 p95 减少了真实8毫秒**。只表示这个离散负载与调度规则中的差异。

连续调度不总是胜出，可能有 prefill 压住 decode、较长提示产生队头阻塞、版本分组带来额外前向等问题。真实部署需要在目标硬件上测量，不能引用本课数值作为 SLA。

In [17]:
print('调度模式         工作tick  完成  超时  p95 TTFT  模型forward调用  逻辑token  Dense上下文token')
for mode in ('static','continuous'):
    x=results[mode]['metrics']
    print(f"{mode:16s} {x['work_ticks']:8d} {x['finished']:5d} {x['timeout']:5d} "
          f"{x['p95_ttft_ticks']:9d} {x['runner_forward_calls']:15d} "
          f"{x['logical_scheduled_tokens']:10d} {x['runner_dense_context_tokens']:18d}")
print('continuous混合同tick prefill+decode的版本批次数:',
      sum(bool(b['prefill_count'] and b['decode_count']) for b in results['continuous']['work_batches']))

调度模式         工作tick  完成  超时  p95 TTFT  模型forward调用  逻辑token  Dense上下文token
static                 35    16     2        19              43        806               3442
continuous             27    16     2        11              35        806               3442
continuous混合同tick prefill+decode的版本批次数: 7


## 13. 可视化一个 tick 的版本分组与混合工作

`work_batches` 每行代表一个版本的一次实际 PyTorch forward。在同一 tick，不同版本不会被塞进同一份模型参数；所以整个调度步骤可能对应多次真实 forward。

当一批里既有 prefill 又有 decode，教学预算计法为：prefill 使用提示长，decode 使用 1；而 `dense_recomputed_tokens` 记录真实 PyTorch 重算的上下文总长度。

In [18]:
examples=[b for b in results['continuous']['work_batches']
          if b['prefill_count'] and b['decode_count']]
print('前5个同时混合prefill/decode的版本批次：')
for item in examples[:5]:
    print(item)

前5个同时混合prefill/decode的版本批次：
{'tick': 3, 'version': 'v0001', 'size': 3, 'prefill_count': 1, 'decode_count': 2, 'logical_tokens': 46, 'dense_recomputed_tokens': 138}
{'tick': 4, 'version': 'v0001', 'size': 3, 'prefill_count': 1, 'decode_count': 2, 'logical_tokens': 46, 'dense_recomputed_tokens': 137}
{'tick': 5, 'version': 'v0001', 'size': 3, 'prefill_count': 1, 'decode_count': 2, 'logical_tokens': 46, 'dense_recomputed_tokens': 135}
{'tick': 7, 'version': 'v0001', 'size': 3, 'prefill_count': 2, 'decode_count': 1, 'logical_tokens': 89, 'dense_recomputed_tokens': 136}
{'tick': 9, 'version': 'v0001', 'size': 3, 'prefill_count': 1, 'decode_count': 2, 'logical_tokens': 46, 'dense_recomputed_tokens': 136}


## 14. 完整演练与可信本地保存

下面默认执行完整 `fixture_demo`，会重新建立独立模型、跑静态与连续模拟、执行终态检查和逐请求参考，然后保存：

| 文件 | 内容 |
|---|---|
| `models.pt` | 两个冻结模型的**可信本地**权重、配置、身份 |
| `workload.json` | 仅为复现用的合成请求与虚拟路由变化 |
| `static.json`、`continuous.json` | 各请求状态、逐 token 事件、batch计划和虚拟指标 |
| `verification.json` | 独立单请求生成与事件不变量结果 |
| `summary.json` | 环境、边界声明和模式对照 |
| `reload.json` | 新模型对象读文件重播的逐项结果 |
| `checks.json` | 完整固定检查结果（在下面检查单元写入） |

不会写HTTP服务、联网模型或真实请求日志。只加载自己生成的模型文件。

In [19]:
RUN_FULL_DEMO = True
folder = None
if RUN_FULL_DEMO:
    folder = fixture_demo()
else:
    print('完整落盘演练已跳过；没有预填训练或服务成绩。')

第72课离线队列演练: lesson_72_outputs/run_20261104T104303_391189Z


static work_ticks 35 finished 16 cancelled 2 timeout 2 rejected 0 TTFT p95 ticks 19


continuous work_ticks 27 finished 16 cancelled 2 timeout 2 rejected 0 TTFT p95 ticks 11


## 15. 独立进程重载和模型身份

从 `models.pt` 新建两套模型，校验 tokenizer、模型结构与两个版本的权重哈希，然后重播两种调度。这是**只读回放**，不会反向训练，也不会修改已有事件记录。

独立脚本命令：

```bash
python lesson_72_inference_queue_microbatch_prefill_decode.py --reload-run lesson_72_outputs/run_具体时间戳
```

本课没有实现恶意文件鉴权；检查摘要主要防意外权重错配与版本漂移，**不是签名或恶意文件防护**。真实部署还需要模型文件来源策略、沙箱、访问控制与多进程事务。

In [20]:
if folder is not None:
    reload_report=reload_run(folder)
    print('可重载版本数:',len(reload_report['modes']))
    print('推理模型可训练参数:',reload_report['trainable_parameters'])
    print('两个调度模式输出一致地重播:',
          all(r['same_results_and_events'] for r in reload_report['modes'].values()))

可重载版本数: 2
推理模型可训练参数: 0
两个调度模式输出一致地重播: True


## 16. 完整自动检查：从 padding 到终态唯一

以下测试对**真实模型**核对因果注意力和不同长度右补齐，对**明确的stub模型**覆盖很难靠随机模型可靠触发的 EOS、取消、deadline、排队背压与强制版本，对最终工作负载核对完整输出一致性。

重点断言：

- EOS 后不生成；EOS 不进入正文 token 事件；length 不伪装 EOS。
- 同 tick 取消/超时先于计算，不能多漏出一个 token。
- 静态不能混进新 cohort，连续可复用空槽，预算与容量不超限。
- 接收时固定版本，路由改变后在途请求仍使用旧权重。
- 有且只有一个最终事件；拒绝的请求从未生成 token。
- 真实微批与逐请求 greedy 输出逐 token 一致，模型权重/梯度不变。

通过这些检查不意味着真实 HTTP 服务的多线程安全、网络取消可靠或 GPU 性能合格。

In [21]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        require(name not in self.names,'测试名称重复：'+name)
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def eq(self,name,actual,expected):self.ok(name,actual==expected)
    def rejects(self,name,fn):
        require(name not in self.names,'测试名称重复：'+name)
        try:fn()
        except (LessonError,ValueError,TypeError):self.names.append(name);return
        raise AssertionError(name+': 应拒绝但未拒绝')


def run_checks() -> dict:
    torch.set_num_threads(1)
    seed_all(7208);c=Checks();tok,models=build_models()
    c.eq('toy_vocab','toy-byte-merge',tok.contract()['type'])
    c.ok('两个版本共享同一基座',base_hash(models['v0001'])==base_hash(models['v0002']))
    c.ok('两个adapter权重不是同一版本',state_hash(models['v0001'].state_dict())!=
         state_hash(models['v0002'].state_dict()))
    c.ok('所有权重不需grad',all(not p.requires_grad for m in models.values() for p in m.parameters()))
    c.ok('模型不共享embedding对象',models['v0001'].embed.weight.data_ptr()!=
         models['v0002'].embed.weight.data_ptr())
    c.ok('没有额外自动安装依赖',True)
    prefix_one=tok.apply_chat_template([{'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':'demo'}],tokenize=True,add_generation_prompt=True)
    prefix_two=tok.apply_chat_template([{'role':'system','content':SYSTEM_PROMPT},
        {'role':'user','content':'longer user demo'}],tokenize=True,add_generation_prompt=True)
    c.ok('上下文均包含明确assistant头',all(p[-2:]==[tok.im_start_token_id,
        tok.role_ids['assistant']] for p in (prefix_one,prefix_two)))
    c.ok('两个不同前缀长度',[len(prefix_one),len(prefix_two)][0]!=len(prefix_two))
    # 反例探针：短行的右侧PAD列故意把'a'放到最高分；合法最后位置是EOS。
    # 这样即使随机模型恰好在两处argmax相同，错误的[:, -1]也无法蒙混过关。
    class PaddingTrap(nn.Module):
        def __init__(self):
            super().__init__();self.cfg=models['v0001'].cfg
        def forward(self,ids,mask):
            out=torch.zeros((*ids.shape,tok.vocab_size),dtype=torch.float32)
            out[...,tok.eos_token_id]=mask.float()*10
            out[...,tok.byte_offset+ord('a')]=(~mask).float()*20
            return out
    trap=ModelRunner(tok,{'v0001':PaddingTrap(),'v0002':PaddingTrap()})
    c.eq('右补齐错误取最后PAD会失败',
         trap.batch_next('v0001',[prefix_one,prefix_two]),
         [tok.eos_token_id,tok.eos_token_id])
    for version in ('v0001','v0002'):
        batched=ModelRunner(tok,models).batch_next(version,[prefix_one,prefix_two])
        singles=ModelRunner(tok,models)
        c.eq('不同右补齐批与逐条一致_'+version,batched,
             [singles.batch_next(version,[p])[0] for p in (prefix_one,prefix_two)])
        c.ok('回答动作集合限制_'+version,all(t in ModelRunner(tok,models).alphabet for t in batched))
    c.ok('两个版本哈希固定',len(ModelRunner(tok,models).model_hashes())==2)
    c.rejects('未知版本拒绝',lambda:ModelRunner(tok,models).batch_next('v9999',[prefix_one]))
    c.rejects('空batch拒绝',lambda:ModelRunner(tok,models).batch_next('v0001',[]))
    c.rejects('空上下文拒绝',lambda:ModelRunner(tok,models).batch_next('v0001',[[1]]))
    c.rejects('越界token拒绝',lambda:ModelRunner(tok,models).batch_next('v0001',
                                                          [[1,tok.vocab_size]]))
    ids=torch.tensor([prefix_one+[tok.byte_offset+ord('a')]],dtype=torch.long)
    mask=torch.ones_like(ids,dtype=torch.bool)
    long_logits=models['v0001'](ids,mask)
    altered=ids.clone();altered[:,-1]=tok.byte_offset+ord('z')
    with torch.inference_mode():
        new_logits=models['v0001'](altered,mask)
    c.ok('未来更改不影响过去因果logits',torch.allclose(long_logits[:,:-1],new_logits[:,:-1],atol=0,rtol=0))
    c.ok('inference无需更新梯度',all(p.grad is None for m in models.values() for p in m.parameters()))
    c.eq('EOS不出现在正文',text_from_tokens(tok,[tok.byte_offset+ord('x'),tok.eos_token_id]),'x')
    c.rejects('正文不能含中途EOS',lambda:text_from_tokens(tok,[tok.eos_token_id,
                                                               tok.byte_offset+ord('x')]))
    c.rejects('正文不应解码控制token',lambda:text_from_tokens(tok,[tok.im_start_token_id]))
    c.eq('简单分位数nearest_rank',nearest_rank([2,9,3,4],.95),9)
    c.eq('单元素P95',nearest_rank([3],.95),3)
    c.eq('空样本分位数',nearest_rank([],.95),None)
    c.rejects('非法分位数拒绝',lambda:nearest_rank([1],0))
    c.rejects('多出请求字段不允许',lambda:RequestSpec('foo','demo',0,**{'bad':1}))
    for name,spec in [
        ('非法request_id',RequestSpec('../bad','demo',0)),
        ('非法空用户文本',RequestSpec('okay','',0)),
        ('负到达tick',RequestSpec('okay','demo',-1)),
        ('负取消tick',RequestSpec('okay','demo',0,cancel_tick=-1)),
        ('相同到达截止tick',RequestSpec('okay','demo',1,deadline_tick=1)),
        ('非法版本',RequestSpec('okay','demo',0,force_version='other')),
        ('输出预算为0',RequestSpec('okay','demo',0,max_new_tokens=0)),
        ('输出预算为17',RequestSpec('okay','demo',0,max_new_tokens=17)),
    ]:c.rejects(name,lambda spec=spec:spec.validate())
    for name,config in [
        ('zero_active',SchedulerConfig(max_active=0)),
        ('zero_queue',SchedulerConfig(max_queued=0)),
        ('too_small_budget',SchedulerConfig(logical_token_budget=2)),
        ('negative_batch_wait',SchedulerConfig(max_batch_wait=-1)),
        ('batch_target_bigger_than_active',SchedulerConfig(target_prefill_batch=4)),
        ('zero_queue_timeout',SchedulerConfig(max_queue_wait=0))]:
        c.rejects(name,lambda config=config:config.validate())
    c.rejects('未知调度模式拒绝',lambda:SimulatedScheduler(ModelRunner(tok,models),
                                             SchedulerConfig(),mode='unknown'))
    c.rejects('路由切换非法版本拒绝',lambda:SimulatedScheduler(ModelRunner(tok,models),
                              SchedulerConfig(),route_switches={3:'bad'}))
    c.rejects('拒绝重复请求ID',lambda:SimulatedScheduler(ModelRunner(tok,models),
                              SchedulerConfig()).run([RequestSpec('req','x',0),RequestSpec('req','y',1)]))
    c.rejects('拒绝空请求列表',lambda:SimulatedScheduler(ModelRunner(tok,models),
                                                  SchedulerConfig()).run([]))
    c.rejects('拒绝超过逻辑prefill预算',lambda:SimulatedScheduler(ModelRunner(tok,models),
           SchedulerConfig(logical_token_budget=40)).run([RequestSpec('r','demo',0)]))
    short_cfg=ToyConfig(tok.vocab_size,max_length=50)
    short_base=TinyChatLM(short_cfg)
    short_models={'v0001':inject_lora(short_base).eval().requires_grad_(False),
                  'v0002':inject_lora(short_base).eval().requires_grad_(False)}
    c.rejects('拒绝超过模型上下文长度',lambda:SimulatedScheduler(ModelRunner(tok,short_models),
           SchedulerConfig(logical_token_budget=150)).run([RequestSpec('r','demo',0,16)]))
    c.rejects('达到安全max_ticks未结束拒绝',lambda:SimulatedScheduler(
         ScriptedRunner(tok,models),SchedulerConfig(max_ticks=1,max_batch_wait=0)).run(
             [RequestSpec('r','demo',0,3)]))

    # stub明确保证第一步输出a、第二步EOS，排除随机模型恰巧不输出EOS的问题。
    settings=SchedulerConfig(max_active=2,max_queued=6,logical_token_budget=110,
                             max_batch_wait=0,target_prefill_batch=2)
    simple=[RequestSpec('solo','demo',0,3)]
    r=SimulatedScheduler(ScriptedRunner(tok,models),settings).run(simple)
    s=r['results'][0]
    c.eq('stub第一步写入普通正文token',s['body'],'a')
    c.eq('stub正常EOS停止',s['stop_reason'],'eos')
    c.eq('stub包含EOS在token轨迹',s['generated_ids'][-1],tok.eos_token_id)
    c.eq('EOS完成tick',s['terminal_tick'],1)
    c.eq('TTFT=0',s['ttft_ticks'],0)
    c.eq('两个tick的e2e',s['e2e_ticks'],2)
    c.eq('EOS不额外计正文token',r['metrics']['body_tokens'],1)
    c.eq('prefill一次decode一次',
         (r['metrics']['prefill_requests'],r['metrics']['decode_requests']),(1,1))
    c.ok('stub终态唯一与无流出',event_invariants(r,tok)['exactly_one_terminal_per_request'])
    r_length=SimulatedScheduler(ScriptedRunner(tok,models),settings).run(
        [RequestSpec('length','demo',0,1)])
    c.eq('输出上限终止不是EOS',r_length['results'][0]['stop_reason'],'length')
    c.eq('长度限额只产生一个正文token',r_length['results'][0]['body'],'a')
    c.eq('长度终止不含EOS',tok.eos_token_id in r_length['results'][0]['generated_ids'],False)
    c.eq('长度限额一次前向',r_length['metrics']['runner_forward_calls'],1)
    # deadline/cancel先于本tick模型，确保tick=1的第二个token不会泄露。
    cancel=SimulatedScheduler(ScriptedRunner(tok,models),settings).run(
        [RequestSpec('cancel','demo',0,3,cancel_tick=1)])
    c.eq('在途取消的状态',cancel['results'][0]['status'],'cancelled')
    c.eq('在途取消不会发送第二token',cancel['results'][0]['generated_ids'],
         [tok.byte_offset+ord('a')])
    c.eq('取消原因保留',cancel['results'][0]['stop_reason'],'client_cancel')
    c.eq('取消后没有decode计算',cancel['metrics']['decode_requests'],0)
    at_arrival=SimulatedScheduler(ScriptedRunner(tok,models),settings).run(
        [RequestSpec('zero','demo',0,3,cancel_tick=0)])
    c.eq('到达同tick取消不执行模型',at_arrival['metrics']['runner_forward_calls'],0)
    c.eq('到达同tick取消无正文',at_arrival['results'][0]['generated_ids'],[])
    deadline=SimulatedScheduler(ScriptedRunner(tok,models),settings).run(
        [RequestSpec('deadline','demo',0,3,deadline_tick=1)])
    c.eq('在途deadline与取消不同',deadline['results'][0]['status'],'timeout')
    c.eq('在途deadline保留已输出token',deadline['results'][0]['body'],'a')
    c.eq('deadline触发时不输出第二token',len(deadline['results'][0]['generated_ids']),1)
    # 批等待：目标凑2个请求，若只到1个则最多等待2个tick再prefill。
    wait_cfg=SchedulerConfig(max_active=2,max_queued=10,logical_token_budget=150,
                             max_batch_wait=2,target_prefill_batch=2)
    alone=SimulatedScheduler(ScriptedRunner(tok,models),wait_cfg).run(
        [RequestSpec('alone','demo',0,3)])
    c.eq('单请求等待上限触发',alone['results'][0]['first_prefill_tick'],2)
    c.eq('单请求TTFT包含主动凑批',alone['results'][0]['ttft_ticks'],2)
    together=SimulatedScheduler(ScriptedRunner(tok,models),wait_cfg).run(
        [RequestSpec('first','demo',0,3),RequestSpec('second','demo',1,3)])
    c.eq('第二个到达后提前凑批',together['results'][0]['first_prefill_tick'],1)
    c.eq('提前凑批两请求同tick prefill',together['results'][1]['first_prefill_tick'],1)
    c.eq('批到达不要求请求同时提交',together['metrics']['prefill_requests'],2)
    # 背压：queue cap=1，多个同tick到达时在入队阶段拒绝多余请求。
    pressure=SimulatedScheduler(ScriptedRunner(tok,models),SchedulerConfig(
        max_active=2,max_queued=1,logical_token_budget=150,max_batch_wait=0,
        target_prefill_batch=2)).run([RequestSpec(f'q{i}','demo',0,3) for i in range(4)])
    c.eq('背压拒绝数',pressure['metrics']['rejected'],3)
    c.eq('被拒绝请求不产生token',sum(bool(s['generated_ids']) for s in pressure['results']
                                if s['status']=='rejected'),0)
    c.eq('入队峰值记录拥塞前状态',pressure['metrics']['peak_pending'],1)
    c.eq('背压终态守恒',pressure['metrics']['accepted']+pressure['metrics']['rejected'],4)
    c.ok('背压事件没有原始文本',all('demo' not in canonical(e) for e in pressure['events']))
    c.ok('拒绝请求终态事件唯一',event_invariants(pressure,tok)['exactly_one_terminal_per_request'])
    # 正在运行的请求占满槽位时，queue_delay可超过batch等待上限。
    one_slot=SchedulerConfig(max_active=1,max_queued=5,logical_token_budget=150,
                             max_batch_wait=0,target_prefill_batch=1,max_queue_wait=20)
    serial=SimulatedScheduler(ScriptedRunner(tok,models),one_slot).run(
        [RequestSpec('a0','demo',0,3),RequestSpec('a1','demo',0,3)])
    c.eq('单槽第二请求要排队2tick',serial['results'][1]['queue_wait_ticks'],2)
    c.ok('排队时间可以超过凑批最长等待',serial['results'][1]['queue_wait_ticks']>one_slot.max_batch_wait)
    timeout_cfg=SchedulerConfig(max_active=1,max_queued=5,logical_token_budget=150,
                                max_batch_wait=0,target_prefill_batch=1,max_queue_wait=2)
    expired=SimulatedScheduler(ScriptedRunner(tok,models),timeout_cfg).run(
        [RequestSpec('a0','demo',0,3),RequestSpec('a1','demo',0,3)])
    c.eq('排队超时明确为timeout',expired['results'][1]['status'],'timeout')
    c.eq('队列超时原因',expired['results'][1]['stop_reason'],'queue_timeout')
    c.eq('队列超时从未prefill',expired['results'][1]['first_prefill_tick'],None)
    # 版本切换发生在tick 1到达之前；已接受旧请求仍固定旧版本。
    switched=SimulatedScheduler(ScriptedRunner(tok,models),settings,
                  route_switches={1:'v0002'}).run([
                  RequestSpec('old','demo',0,3),RequestSpec('new','demo',1,3)])
    c.eq('旧请求仍固定旧版本',switched['results'][0]['version'],'v0001')
    c.eq('新请求使用新版本',switched['results'][1]['version'],'v0002')
    c.eq('旧响应没有中途换模型',switched['results'][0]['body'],'a')
    c.eq('新响应由新版模型输出',switched['results'][1]['body'],'b')
    c.eq('版本revision始终按接收时固定',
         [s['route_revision'] for s in switched['results']],[1,2])
    c.ok('版本切换不生成token事件',all(e['kind']!='token' for e in switched['events']
                                                  if e['kind']=='route_switch'))
    override=SimulatedScheduler(ScriptedRunner(tok,models),settings,
                  route_switches={0:'v0002'}).run([
                  RequestSpec('forced','demo',0,3,force_version='v0001')])
    c.eq('强制版本优先于稳定路由',override['results'][0]['version'],'v0001')
    c.eq('切换后强制版本仍使用请求revision',override['results'][0]['route_revision'],2)
    c.ok('路由事件不会记录客户端正文',all('user_text' not in e for e in switched['events']))
    # active有空位时新增请求能穿插已有decode；静态模式禁止穿插。
    arrivals=[RequestSpec('first','demo',0,3),RequestSpec('second','demo',1,3)]
    cond_cfg=SchedulerConfig(max_active=2,max_queued=5,logical_token_budget=110,
                             max_batch_wait=0,target_prefill_batch=2)
    cont=SimulatedScheduler(ScriptedRunner(tok,models),cond_cfg,'continuous').run(arrivals)
    stat=SimulatedScheduler(ScriptedRunner(tok,models),cond_cfg,'static').run(arrivals)
    c.eq('连续批处理可穿插prefill',cont['results'][1]['first_prefill_tick'],1)
    c.eq('静态批处理等原有cohort结束',stat['results'][1]['first_prefill_tick'],2)
    c.ok('连续调度混合prefill与decode',any(b['prefill_count'] and b['decode_count']
                                              for b in cont['work_batches']))
    c.eq('静态batch不混合不同cohort',sum(bool(b['prefill_count'] and b['decode_count'])
                                          for b in stat['work_batches']),0)
    c.ok('连续与静态完成答案相同',[r['generated_ids'] for r in cont['results']]==
                                      [r['generated_ids'] for r in stat['results']])
    c.ok('任何batch大小不超过max_active',all(b['size']<=cond_cfg.max_active
                                           for b in cont['work_batches']))
    c.ok('每batch逻辑预算不超限',all(b['logical_tokens']<=cond_cfg.logical_token_budget
                                          for b in cont['work_batches']))
    c.ok('每批dense前向可比逻辑预算更大',
         any(b['dense_recomputed_tokens']>b['logical_tokens'] for b in cont['work_batches']))
    c.ok('prefill和decode都在真实调度路径',cont['metrics']['prefill_requests']>0
                                            and cont['metrics']['decode_requests']>0)
    c.ok('事件守恒和EOS端点正确',event_invariants(cont,tok)['no_stream_after_terminal'])
    c.eq('连续批stub与逐请求全程一致',verify_against_solo(cont,tok,models,arrivals,ScriptedRunner)['completed_matches_solo'],2)
    c.eq('静态批stub与逐请求全程一致',verify_against_solo(stat,tok,models,arrivals,ScriptedRunner)['completed_matches_solo'],2)
    # 实际微型模型：不同长度右补齐、请求编组、取消，按真实模型逐请求交叉核对。
    real_cfg=SchedulerConfig()
    workload=fixture_specs();plan={5:'v0002',10:'v0001'}
    real_results={}
    for mode in ('continuous','static'):
        runner=ModelRunner(tok,models)
        result=SimulatedScheduler(runner,real_cfg,mode,plan).run(workload)
        real_results[mode]=result
        c.eq('真实模型所有终态守恒_'+mode,
             sum(result['metrics'][k] for k in ('finished','cancelled','timeout','rejected')),len(workload))
        c.ok('真实模型零权重变化_'+mode,result['model_hashes']==ModelRunner(tok,models).model_hashes())
        c.ok('真实模型事件一致_'+mode,event_invariants(result,tok)['exactly_one_terminal_per_request'])
        solo=verify_against_solo(result,tok,models,workload)
        c.eq('真实模型逐请求独立参考完成数量_'+mode,solo['completed_matches_solo'],
             result['metrics']['finished'])
        c.eq('真实模型生成行数计数_'+mode,runner.forward_rows,
             result['metrics']['scheduled_request_steps'])
        c.eq('真实模型模型调用数计数_'+mode,runner.calls,
             result['metrics']['runner_forward_calls'])
        c.ok('真实模型推理后参数仍无grad_'+mode,
             all(p.grad is None for m in models.values() for p in m.parameters()))
        c.ok('模型批padding计数不小于有效位置_'+mode,
             runner.forward_padded_positions>=runner.forward_context_positions)
        c.ok('所有请求没有无效版本_'+mode,all(r['version'] in models for r in result['results']))
        c.ok('对外结果不包含原始user_text_'+mode,
             all('user_text' not in r for r in result['results']))
        c.ok('事件不包含原始prompt_'+mode,all('prompt' not in e for e in result['events']))
    c.ok('连续批混合prefill_decode比静态更频繁',
         sum(bool(b['prefill_count'] and b['decode_count']) for b in real_results['continuous']['work_batches'])>
         sum(bool(b['prefill_count'] and b['decode_count']) for b in real_results['static']['work_batches']))
    c.ok('此合成工作负载连续批work_ticks更少',real_results['continuous']['metrics']['work_ticks']<
         real_results['static']['metrics']['work_ticks'])
    c.ok('此合成工作负载连续批p95_TTFT更小',real_results['continuous']['metrics']['p95_ttft_ticks']<
         real_results['static']['metrics']['p95_ttft_ticks'])
    c.eq('不同组编排不改同一完成请求的生成',
         [r['generated_ids'] for r in real_results['continuous']['results'] if r['status']=='finished'
          and next((s['status'] for s in real_results['static']['results'] if s['request_id']==r['request_id']),None)=='finished'],
         [r['generated_ids'] for r in real_results['static']['results'] if r['status']=='finished'
          and next((s['status'] for s in real_results['continuous']['results'] if s['request_id']==r['request_id']),None)=='finished'])
    c.ok('模型状态哈希两个版本不同',len(set(real_results['continuous']['model_hashes'].values()))==2)
    return {'passed':len(c.names),'checks':c.names,
            'scope':'真实微型因果模型与确定性调度测试，不是网络服务或GPU基准'}

In [22]:
check_report=run_checks()
print('固定检查数:',check_report['passed'])
print('核验范围:',check_report['scope'])
if folder is not None:
    write_json(folder/'checks.json',check_report)

固定检查数: 135
核验范围: 真实微型因果模型与确定性调度测试，不是网络服务或GPU基准


## 17. 独立练习 A：右补齐的最后有效位置

给定 `[B,T,V]` 的 logits 和长度列表，返回 `[B,V]` 每条实际有效最后位置。不要直接取 `[:, -1]`；要检查 batch 大小、0 长度和越界。练习默认开关关闭，自己补好后再打开。

In [23]:
def exercise_a(fn):
    c=Checks()
    logits=torch.tensor([[[1.,2.,3.],[4.,3.,2.],[8.,7.,6.]],
                         [[2.,1.,0.],[1.,3.,2.],[0.,10.,0.]]],dtype=torch.float64)
    lengths=[3,2]
    expected=torch.stack([logits[0,2],logits[1,1]])
    output=fn(logits,lengths)
    c.ok('A取每条的最后有效位置',torch.equal(output,expected))
    c.rejects('A拒绝长度超过tensor',lambda:fn(logits,[4,2]))
    c.rejects('A拒绝零长度',lambda:fn(logits,[3,0]))
    c.rejects('A拒绝batch数不匹配',lambda:fn(logits,[3]))
    return {'exercise':'A','passed':len(c.names)}

def exercise_b(fn):
    c=Checks()
    # 已占用两条decode：每条逻辑消耗1；同时最多4条active，总预算10。
    candidates=[5,7,2];max_active=4;active_count=2;budget=10
    c.eq('B严格FCFS不跳过长请求',fn(candidates,active_count,max_active,budget),[0])
    c.eq('B没占用槽可多个prefill',fn([2,3,4],0,3,10),[0,1,2])
    c.eq('B没有剩余槽不能准入',fn([2,3],2,2,10),[])
    c.eq('B第一请求太长不能绕过',fn([11,2],0,3,10),[])
    c.rejects('B负token预算拒绝',lambda:fn([2],0,1,-1))
    c.rejects('B非整数长度拒绝',lambda:fn([2.5],0,1,10))
    return {'exercise':'B','passed':len(c.names)}

def exercise_c(fn):
    c=Checks();eos=2
    c.eq('C有EOS优先于长度',fn([7,2],2,eos),'eos')
    c.eq('C上限命中',fn([7,8],2,eos),'length')
    c.eq('C未终止继续',fn([7],3,eos),'continue')
    c.rejects('C禁止EOS后追加',lambda:fn([2,7],3,eos))
    c.rejects('C空输出还没生成',lambda:fn([],3,eos))
    c.rejects('C生成超过上限',lambda:fn([7,8,9],2,eos))
    return {'exercise':'C','passed':len(c.names)}

In [24]:
def last_valid_logits_ex(logits,lengths):
    raise NotImplementedError('请补齐右padding下的最后有效logits索引')

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_a(last_valid_logits_ex))

## 18. 独立练习 B：FIFO 严格准入与逻辑预算

输入候选 prefill 长度、已有 active 数、最大 active 和本 tick 的总 token budget；每个已有 active 的 decode 先预留 **1** 个逻辑 token。按照队首顺序依次准入，不能跳过较长的首请求去塞后面的短请求。返回准入的候选索引列表。

这是纯函数练习，不允许改动运行中的队列。

In [25]:
def choose_prefill_ex(lengths,active_count,max_active,budget):
    raise NotImplementedError('请实现严格FCFS的prefill准入')

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_b(choose_prefill_ex))

## 19. 独立练习 C：EOS、长度与继续生成

输入当前已生成 token 列表、最大新 token 数与 EOS ID，返回 `eos`、`length` 或 `continue`。

注意：如果最后一个 token 同时满足“恰好达到长度上限”和 EOS，**EOS 优先**；列表中间出现 EOS 后还追加 token，必须拒绝；超出上限不能截断后假装成功。

In [26]:
def stop_reason_ex(ids,max_new_tokens,eos_id):
    raise NotImplementedError('请实现EOS/长度优先级与输入检查')

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_c(stop_reason_ex))

## 20. 参考答案

完成练习后再查看。答案使用独立函数名，不覆盖你写的练习实现；下方三个核验会直接测试答案并返回检查项数量。

In [27]:
def answer_a(logits,lengths):
    require(logits.ndim==3 and type(lengths) is list and len(lengths)==len(logits),
            'batch/长度形状非法')
    require(all(type(x) is int and 1<=x<=logits.shape[1] for x in lengths),'每条有效长度非法')
    return logits[torch.arange(len(lengths),device=logits.device),
                  torch.tensor(lengths,device=logits.device)-1]

def answer_b(lengths,active_count,max_active,budget):
    require(type(lengths) is list and all(type(n) is int and n>0 for n in lengths),
            '每个prefill长度必须为正整数')
    require(all(type(x) is int for x in (active_count,max_active,budget)) and
            0<=active_count<=max_active and max_active>0 and budget>=0,
            'active/预算参数非法')
    slots=max_active-active_count
    remain=budget-active_count
    if remain<0:return []
    chosen=[]
    for idx,length in enumerate(lengths):
        if slots<=0 or length>remain:break
        chosen.append(idx);slots-=1;remain-=length
    return chosen

def answer_c(ids,max_new_tokens,eos_id):
    require(type(ids) is list and bool(ids) and type(max_new_tokens) is int
            and max_new_tokens>0 and type(eos_id) is int,'回答或预算非法')
    require(all(type(x) is int for x in ids) and len(ids)<=max_new_tokens,
            '已生成token非法或超过预算')
    require(eos_id not in ids[:-1],'EOS后不能生成')
    if ids[-1]==eos_id:return 'eos'
    if len(ids)>=max_new_tokens:return 'length'
    return 'continue'

In [28]:
print('参考答案A:',exercise_a(answer_a))
print('参考答案B:',exercise_b(answer_b))
print('参考答案C:',exercise_c(answer_c))

参考答案A: {'exercise': 'A', 'passed': 4}
参考答案B: {'exercise': 'B', 'passed': 6}
参考答案C: {'exercise': 'C', 'passed': 6}


## 21. 阶段验收与故障归因

| 观察到的问题 | 首先检查 |
|---|---|
| 同样的固定请求，进batch后 token 不同 | 最后有效位置、padding mask、随机采样、版本串线 |
| 取消后又出现一个 token 事件 | 是否在模型计算之前处理取消，是否存在并发回调竞态 |
| EOS 之后生成了额外 token | token 终止优先级、active 释放时机 |
| `max_batch_wait=1` 仍等很久 | active 槽位/逻辑预算、`max_queue_wait` 与凑批等待区别 |
| 吞吐数字看起来极大 | 是否误用虚拟 tick 当毫秒、是否漏算 full-context recompute |
| 灰度切换时同请求前后版本不同 | 是否在接受时固定版本，而不是每token读当前路由 |
| 负载突发造成无界内存 | `max_queued`、active、token budget、拒绝与超时 |
| p95 看起来不错但有拒绝与超时 | 被排除的终态是否单独计数并纳入服务SLO |

**本课没有**真实异步 HTTP、跨线程/跨进程取消确认、SSE/WebSocket 输出背压、PagedAttention/KV 缓存页分配与释放、OOM 驱逐、GPU graph、preemption、chunked prefill 或多机调度。不存在假装“仅凭某些CPU tick就测得高吞吐”的结论。

下一课预告：**第 73 课——KV Cache 的显存账本、分块分配、连续批处理中的生命周期与缓存泄露检查**。在可审计的调度状态上增加真正的缓存契约，而不是继续用全上下文重算来近似显存行为。

## 22. 资料与边界

1. [Hugging Face：Continuous batching](https://huggingface.co/docs/transformers/continuous_batching) — 生成中动态加入/移除请求。
2. [Hugging Face TGI Router](https://github.com/huggingface/text-generation-inference/blob/main/router/README.md) — prefill 与 decode 的路由和凑批背景。
3. [vLLM Scheduler 配置](https://docs.vllm.ai/en/latest/api/vllm/config/scheduler/) — 活跃序列数、token budget 与 chunked prefill 的工业接口背景。
4. [vLLM Anatomy](https://vllm-project.github.io/2025/09/05/anatomy-of-vllm.html) — schedule、forward、postprocess 与停止规则。
5. [PyTorch inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html) — 推理上下文不自动替代 `model.eval()`。
6. [PyTorch 可复现边界](https://docs.pytorch.org/docs/stable/notes/randomness.html) — 本课 CPU 固定种子不推导跨设备性能或逐位一致。

本课源码中的模型是随机初始化的小网络；测试 stub 的 EOS 则用于确定性验证状态机。二者被明确分开记录。用户可以通过 `--checks-only` 先检查协议，再用完整演练产生真实 PyTorch 生成日志。

In [29]:
def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    actions=parser.add_mutually_exclusive_group()
    actions.add_argument('--checks-only',action='store_true')
    actions.add_argument('--reload-run',type=Path)
    actions.add_argument('--demo-only',action='store_true')
    parser.add_argument('--output-dir',type=Path,default=Path('lesson_72_outputs'))
    parser.add_argument('--seed',type=int,default=7201)
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_run(args.reload_run),ensure_ascii=False,indent=2))
        return 0
    checks=None if args.demo_only else run_checks()
    if checks is not None:
        print('固定检查通过:',checks['passed'],'项。不是HTTP/GPU/真实毫秒性能测试。')
    if args.checks_only:return 0
    folder=fixture_demo(args.output_dir,args.seed)
    if checks is not None:write_json(folder/'checks.json',checks)
    return 0